# 01 Folds and data

Builds the common site-day population that every method is scored on and the eighteen leave-one-station-out folds, and writes the fold manifest that proves the station exclusions. This is the first notebook of the Phase 3 final evaluation; every later notebook reads its files.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** `dataset/final/dataset_alpha.parquet` and `dataset_beta.parquet` (frozen, hashes verified against `config/final_evaluation.yaml`).

**Outputs.** `outputs/01_final_evaluation/01_folds/`: `site_days_alpha.parquet`, `site_days_beta.parquet` (one row per station-date with completeness, confidence and the labelled window), `fold_manifest.csv`, `population.csv`; the manifest `manifests/01_folds.json`.

**Approximate runtime.** About one minute.

**Prerequisites.** None.

**Main process.**

1. Load the configuration and verify the dataset hashes.
2. Build the site-day index per cohort; a day enters the evaluation only when all 96 quarter-hours are present.
3. Build the folds: each station held out once; M8 trains on all other stations of both cohorts (Beta `sure` days only); M9 calibrates on the other stations of the same cohort.
4. Write the fold manifest and check it: no station twice, none in its own training set.

## 1. Setup

Locate the article folder, import the evaluation package and load the configuration. Loading the configuration verifies the SHA-256 of every dataset, so a wrong or edited data file stops the run here.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Population and folds

The stage function does the work; the notebook shows what it produced. `population.csv` records how many site-days each cohort has, how many are complete, and how many complete headline-confidence days carry a labelled RPF error. `fold_manifest.csv` lists, for every fold, the held-out station, the M8 training stations, the M9 calibration stations and a hash of the exact training keys.

In [ ]:
result = cli.stage_folds(SETTINGS)
display(result["population"])

In [ ]:
manifest = result["manifest"]
display(manifest[["fold_id", "cohort", "held_out", "n_heldout_days", "n_heldout_headline_days",
                  "n_heldout_rpf_days", "n_m8_training_days", "n_m8_training_rpf_days", "n_m9_calibration_days"]])

## 3. Checks

The same assertions the tests make, run on the real manifest: every station is held out exactly once and never appears in its own training or calibration set.

In [ ]:
from final_eval.folds import check_manifest

check_manifest(manifest)
assert len(manifest) == 18
assert manifest["held_out"].is_unique
print("Fold manifest valid: 18 folds, every station held out once.")

## Conclusion

The population and the folds are fixed. The Alpha cohort contributes ten held-out stations and the Beta cohort eight; the counts above are the denominators of every headline number. Continue with notebook 02 (M8 training, heavy) or, to reproduce the M7 and M9 results without M8, with notebooks 03 and 05.